In [ ]:
#new layer
import arcpy
import os

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
source_fc = os.path.join(gdb, "SEFM_events_94_24")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matching_complete")

# Enable overwrite capability in case I need to re-run
arcpy.env.overwriteOutput = True

print(f"Creating a fresh copy of master events dataset...")
print(f"Source: {os.path.basename(source_fc)}")
print(f"Target: {os.path.basename(target_fc)}...")

# Execute the copy feature class process
arcpy.management.CopyFeatures(source_fc, target_fc)

# Clear workspace cache to force ArcGIS Pro to display the new layer immediately
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Created layer: SEFM_events_94_24_matching_complete")
print("Refresh Geodatabase catalog in ArcGIS Pro to see it.")
print("="*60)


In [ ]:
#create simplified fields for mtbs and landfire. Fill mtbs_simplified
import arcpy
import os

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matching_complete")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE FIELDS EXIST ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
fields_to_add = []

if "mtbs_simplified" not in existing_fields:
    fields_to_add.append(["mtbs_simplified", "TEXT", "mtbs_simplified"])
if "landfire_simplified" not in existing_fields:
    fields_to_add.append(["landfire_simplified", "TEXT", "landfire_simplified"])

if fields_to_add:
    print("Step 1: Creating clean snake_case fields...")
    # Passing name as both field name and alias forces Pro to keep snake_case display
    arcpy.management.AddFields(target_fc, fields_to_add)
else:
    print("Step 1: Simplified fields already exist. Proceeding to update values...")

# --- 2. POPULATE MTBS_SIMPLIFIED ---
print("Step 2: Processing MTBS mapping rules...")
update_count = 0

# Adjust 'mtbs_match' field name if it is spelled differently in feature class
fields = ["mtbs_match", "mtbs_simplified"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        mtbs_raw = row[0]
        
        # Handle cases where the field is empty, a space, or a database NULL
        if mtbs_raw is None or str(mtbs_raw).strip() == "" or str(mtbs_raw).lower() == "none":
            row[1] = None
        else:
            # Clean string matching to avoid case-sensitivity errors
            val = str(mtbs_raw).strip().lower()
            
            if val == "prescribed fire":
                row[1] = "prescribed"
            elif val in ["wildfire", "wildland fire use"]:
                row[1] = "wildfire"
            elif val == "unknown":
                row[1] = "unknown"
            else:
                # If there's an unforeseen classification string, keep it raw or map to unknown
                row[1] = "unknown"
                
        cur.updateRow(row)
        update_count += 1

# Flush cache so ArcGIS Pro visualizes the changes immediately
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Created both simplified columns.")
print(f"-> Evaluated and updated {update_count:,} records in mtbs_simplified.")
print("="*60)

In [ ]:
# complete landfire_simplified
import arcpy
import os

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matching_complete")

print("Processing Landfire mapping rules...")
update_count = 0

# Adjust 'landfire_match' field name if it is spelled differently in feature class
fields = ["landfire_match", "landfire_simplified"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        lf_raw = row[0]
        
        # Handle cases where the field is empty, a space, or a database NULL
        if lf_raw is None or str(lf_raw).strip() == "" or str(lf_raw).lower() == "none":
            row[1] = None
        else:
            # Clean string matching to avoid case-sensitivity errors
            val = str(lf_raw).strip().lower()
            
            if val == "prescribed fire":
                row[1] = "prescribed"
            elif val in ["wildfire", "wildland fire use"]:
                row[1] = "wildfire"
            elif val in ["wildland fire", "unknown"]:
                row[1] = "unknown"
            else:
                # Catch-all for any other unexpected strings
                row[1] = "unknown"
                
        cur.updateRow(row)
        update_count += 1

# Flush cache so ArcGIS Pro visualizes the changes immediately
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Updated {update_count:,} records in landfire_simplified.")
print("="*60)

In [ ]:
#create classification, apply hierarchy
import arcpy
import os

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matching_complete")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE FINAL FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "classification" not in existing_fields:
    print("Step 1: Adding clean 'classification' field...")
    arcpy.management.AddFields(target_fc, [["classification", "TEXT", "classification"]])
else:
    print("Step 1: 'classification' field already exists. Proceeding to update...")

# --- 2. RUN THE HIERARCHICAL EVALUATION LOOP ---
print("Step 2: Running hierarchical evaluation loop...")
update_count = 0

# Specify all the necessary fields in the exact priority order
fields = [
    "mtbs_simplified",       # Index 0
    "fod_match",             # Index 1
    "landfire_simplified",   # Index 2
    "permit_match",          # Index 3
    "classification"         # Index 4 (Target)
]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        # Extract fields and normalize them safely to lowercase strings or None
        mtbs   = str(row[0]).strip().lower() if row[0] not in [None, "", "None"] else None
        fod    = str(row[1]).strip().lower() if row[1] not in [None, "", "None"] else None
        lf     = str(row[2]).strip().lower() if row[2] not in [None, "", "None"] else None
        permit = str(row[3]).strip().lower() if row[3] not in [None, "", "None"] else None
        
        # --- THE HIERARCHY ---
        
        # Priority 1: MTBS (wildfire or prescribed)
        if mtbs in ["wildfire", "prescribed"]:
            row[4] = mtbs
            
        # Priority 2: FOD (wildfire)
        elif fod == "wildfire":
            row[4] = "wildfire"
            
        # Priority 3: Landfire (wildfire or prescribed)
        elif lf in ["wildfire", "prescribed"]:
            row[4] = lf
            
        # Priority 4: Permits (prescribed)
        # Note: We match your previous cell's permit string designation "prescribed_fire"
        elif permit in ["prescribed", "prescribed_fire"]:
            row[4] = "prescribed"
            
        # Default Baseline: If unmatched anywhere, keep it NULL
        else:
            row[4] = None
            
        cur.updateRow(row)
        update_count += 1

# Flush workspace cache to push visualization to ArcGIS Pro
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Prioritization pipeline complete.")
print(f"-> Evaluated and classified {update_count:,} events in 'classification'.")
print("="*60)

In [ ]:
#create matched only layer so I can figure out date
import arcpy
import os

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
source_fc = os.path.join(gdb, "SEFM_events_94_24_matching_complete")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

print(f"Extracting matched events where classification is not NULL...")
print(f"Source: {os.path.basename(source_fc)}")
print(f"Target: {os.path.basename(target_fc)}...")

# SQL expression to isolate matched data
where_clause = "classification IS NOT NULL"

# Execute the selection and save as a new feature class
arcpy.analysis.Select(
    in_features=source_fc,
    out_feature_class=target_fc,
    where_clause=where_clause
)

# Clear workspace cache to force ArcGIS Pro to display the new layer immediately
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Created sandbox layer: SEFM_events_94_24_matched_only")
print("Refresh Geodatabase catalog in ArcGIS Pro to see it.")
print("="*60)

In [ ]:
#Pull record_date from record that assigned classification using hierarchy. 
import arcpy
import os

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE RECORD_DATE FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "record_date" not in existing_fields:
    print("Step 1: Adding clean 'record_date' DATE field...")
    arcpy.management.AddFields(target_fc, [["record_date", "DATE", "record_date"]])
else:
    print("Step 1: 'record_date' field already exists. Proceeding to update...")

# --- 2. RUN THE DATE HIERARCHICAL EXTRACTION ---
print("Step 2: Executing date extraction hierarchy...")
update_count = 0

# Order our fields exactly to match the logic flow
fields = [
    "mtbs_simplified",       # Index 0
    "mtbs_ig_date",          # Index 1
    "fod_match",             # Index 2
    "fod_discovery_date",    # Index 3
    "landfire_simplified",   # Index 4
    "landfire_start_date",   # Index 5
    "permit_match",          # Index 6
    "permit_start_date",     # Index 7
    "record_date"            # Index 8 (Target)
]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        # Normalize simplified text states to string/None for clean checking
        mtbs_sim = str(row[0]).strip().lower() if row[0] not in [None, "", "None"] else None
        fod_mat  = str(row[2]).strip().lower() if row[2] not in [None, "", "None"] else None
        lf_sim   = str(row[4]).strip().lower() if row[4] not in [None, "", "None"] else None
        p_mat    = str(row[6]).strip().lower() if row[6] not in [None, "", "None"] else None
        
        # --- DOMINO DATE EXTRACTION ---
        
        # Step 1: MTBS wins
        if mtbs_sim in ["wildfire", "prescribed"]:
            row[8] = row[1]
            
        # Step 2: FOD wins
        elif fod_mat == "wildfire":
            row[8] = row[3]
            
        # Step 3: Landfire wins
        elif lf_sim in ["wildfire", "prescribed"]:
            row[8] = row[5]
            
        # Step 4: Permits win
        elif p_mat in ["prescribed", "prescribed_fire"]:
            row[8] = row[7]
            
        # Fallback security anchor
        else:
            row[8] = None
            
        cur.updateRow(row)
        update_count += 1

# Force ArcGIS Pro to refresh its data visualization layout
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Date alignment complete.")
print(f"-> Assigned baseline temporal records to {update_count:,} features.")
print("="*60)

In [ ]:
#calculate days from pre_bd_min
import arcpy
import os
from datetime import datetime

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE INT FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "days_MIN_prebd_min" not in existing_fields:
    print("Step 1: Adding 'days_MIN_prebd_min' integer field...")
    arcpy.management.AddFields(target_fc, [["days_MIN_prebd_min", "LONG", "days_MIN_prebd_min"]])
else:
    print("Step 1: Field already exists. Proceeding to calculation...")

# --- 2. CALCULATE DAYS INTERVAL ---
print("Step 2: Calculating days between satellite start and record date...")
update_count = 0
missing_dates_count = 0

fields = ["MIN_prebd_min_corrected", "record_date", "days_MIN_prebd_min"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        sefm_raw    = row[0]
        record_date = row[1]
        
        # Check if both fields have valid data
        if sefm_raw in [None, "", 0, "None"] or record_date is None:
            row[2] = None
            missing_dates_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # 1. Parse YYYYMMDD string/int to Python date object
            sefm_str = str(int(float(sefm_raw))).strip() # Clean float scientific notation artifacts if any
            sefm_date = datetime.strptime(sefm_str, "%Y%m%d").date()
            
            # 2. Extract date-only object from the record_date timestamp
            if isinstance(record_date, datetime):
                rec_date_only = record_date.date()
            else:
                rec_date_only = record_date
                
            # 3. Calculate difference: record_date minus satellite start window
            delta = rec_date_only - sefm_date
            row[2] = delta.days
            update_count += 1
            
        except Exception as e:
            # Catch parsing anomalies safely without failing the whole execution loop
            row[2] = None
            missing_dates_count += 1
            
        cur.updateRow(row)

# Clear cache to push modifications directly to Pro display
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Day offset calculation complete.")
print(f"-> Calculated intervals: {update_count:,} rows")
print(f"-> Records skipped (NULL dates/errors): {missing_dates_count:,} rows")
print("="*60)

In [ ]:
#calculate days from MAX_bd_min_plus8
import arcpy
import os
from datetime import datetime

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE INT FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "days_MAX_bd_min_plus8" not in existing_fields:
    print("Step 1: Adding 'days_MAX_bd_min_plus8' integer field...")
    arcpy.management.AddFields(target_fc, [["days_MAX_bd_min_plus8", "LONG", "days_MAX_bd_min_plus8"]])
else:
    print("Step 1: Field already exists. Proceeding to calculation...")

# --- 2. CALCULATE DAYS INTERVAL ---
print("Step 2: Calculating days between satellite end (+8) and record date...")
update_count = 0
missing_dates_count = 0

fields = ["MAX_bd_min_corrected_plus8", "record_date", "days_MAX_bd_min_plus8"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        sefm_raw    = row[0]
        record_date = row[1]
        
        # Check if both fields have valid data
        if sefm_raw in [None, "", 0, "None"] or record_date is None:
            row[2] = None
            missing_dates_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # 1. Parse YYYYMMDD string/int to Python date object
            sefm_str = str(int(float(sefm_raw))).strip() # Clean float scientific notation artifacts if any
            sefm_date = datetime.strptime(sefm_str, "%Y%m%d").date()
            
            # 2. Extract date-only object from the record_date timestamp
            if isinstance(record_date, datetime):
                rec_date_only = record_date.date()
            else:
                rec_date_only = record_date
                
            # 3. Calculate difference: record_date minus satellite end window (+8)
            delta = rec_date_only - sefm_date
            row[2] = delta.days
            update_count += 1
            
        except Exception as e:
            # Catch parsing anomalies safely without failing the whole execution loop
            row[2] = None
            missing_dates_count += 1
            
        cur.updateRow(row)

# Clear cache to push modifications directly to Pro display
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. End-window day offset calculation complete.")
print(f"-> Calculated intervals: {update_count:,} rows")
print(f"-> Records skipped (NULL dates/errors): {missing_dates_count:,} rows")
print("="*60)

In [ ]:
#calculate days from MIN_bdmin_corrected
import arcpy
import os
from datetime import datetime

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE INT FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "days_MIN_bd_min" not in existing_fields:
    print("Step 1: Adding 'days_MIN_bd_min' integer field...")
    arcpy.management.AddFields(target_fc, [["days_MIN_bd_min", "LONG", "days_MIN_bd_min"]])
else:
    print("Step 2: Field already exists. Proceeding to calculation...")

# --- 2. CALCULATE DAYS INTERVAL ---
print("Step 2: Calculating days between satellite burn window start and record date...")
update_count = 0
missing_dates_count = 0

fields = ["MIN_bd_min_corrected", "record_date", "days_MIN_bd_min"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        sefm_raw    = row[0]
        record_date = row[1]
        
        # Check if both fields have valid data
        if sefm_raw in [None, "", 0, "None"] or record_date is None:
            row[2] = None
            missing_dates_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # 1. Parse YYYYMMDD string/int to Python date object
            sefm_str = str(int(float(sefm_raw))).strip() # Clean float scientific notation artifacts if any
            sefm_date = datetime.strptime(sefm_str, "%Y%m%d").date()
            
            # 2. Extract date-only object from the record_date timestamp
            if isinstance(record_date, datetime):
                rec_date_only = record_date.date()
            else:
                rec_date_only = record_date
                
            # 3. Calculate difference: record_date minus satellite burn window start
            delta = rec_date_only - sefm_date
            row[2] = delta.days
            update_count += 1
            
        except Exception as e:
            # Catch parsing anomalies safely without failing the whole execution loop
            row[2] = None
            missing_dates_count += 1
            
        cur.updateRow(row)

# Clear cache to push modifications directly to Pro display
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Core burn-window day offset calculation complete.")
print(f"-> Calculated intervals: {update_count:,} rows")
print(f"-> Records skipped (NULL dates/errors): {missing_dates_count:,} rows")
print("="*60)

In [ ]:
#find midpoint between MIN_prebd_min and MAX_bdmin
import arcpy
import os
from datetime import datetime

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE DATE FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "mid_prebd_MAX_bd_min" not in existing_fields:
    print("Step 1: Adding 'mid_prebd_MAX_bd_min' DATE field...")
    arcpy.management.AddFields(target_fc, [["mid_prebd_MAX_bd_min", "DATE", "mid_prebd_MAX_bd_min"]])
else:
    print("Step 1: Field already exists. Proceeding to calculation...")

# --- 2. CALCULATE MIDPOINT DATE ---
print("Step 2: Calculating chronological midpoint between satellite windows...")
update_count = 0
skipped_count = 0

fields = ["MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", "mid_prebd_MAX_bd_min"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        start_raw = row[0]
        end_raw   = row[1]
        
        # Check for missing/empty values in either bound
        if start_raw in [None, "", 0, "None"] or end_raw in [None, "", 0, "None"]:
            row[2] = None
            skipped_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # 1. Parse YYYYMMDD format safely to datetime objects
            start_str = str(int(float(start_raw))).strip()
            end_str   = str(int(float(end_raw))).strip()
            
            start_date = datetime.strptime(start_str, "%Y%m%d")
            end_date   = datetime.strptime(end_str, "%Y%m%d")
            
            # 2. Compute the exact time delta and divide it by 2
            window_delta = end_date - start_date
            midpoint_date = start_date + (window_delta / 2)
            
            # 3. Assign the datetime object to the target DATE field
            row[2] = midpoint_date
            update_count += 1
            
        except Exception as e:
            # Fallback for any parsing string anomalies
            row[2] = None
            skipped_count += 1
            
        cur.updateRow(row)

# Flush workspace cache to push the date updates directly to ArcGIS Pro
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Midpoint date compilation complete.")
print(f"-> Populated true calendar midpoints: {update_count:,} rows")
print(f"-> Records skipped due to NULL bounds: {skipped_count:,} rows")
print("="*60)

In [ ]:
#calculate days from midpoint just created
import arcpy
import os
from datetime import datetime

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE INT FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "days_mid_prebd_MAX_bd_min" not in existing_fields:
    print("Step 1: Adding 'days_mid_prebd_MAX_bd_min' integer field...")
    arcpy.management.AddFields(target_fc, [["days_mid_prebd_MAX_bd_min", "LONG", "days_mid_prebd_MAX_bd_min"]])
else:
    print("Step 1: Field already exists. Proceeding to calculation...")

# --- 2. CALCULATE DAYS INTERVAL ---
print("Step 2: Calculating days between agency record date and satellite window midpoint...")
update_count = 0
missing_dates_count = 0

fields = ["record_date", "mid_prebd_MAX_bd_min", "days_mid_prebd_MAX_bd_min"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        record_date   = row[0]
        midpoint_date = row[1]
        
        # Check if both fields have valid data
        if record_date is None or midpoint_date is None:
            row[2] = None
            missing_dates_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # Clean up potential datetime/date type mismatches natively
            rec_d = record_date.date() if isinstance(record_date, datetime) else record_date
            mid_d = midpoint_date.date() if isinstance(midpoint_date, datetime) else midpoint_date
                
            # Calculate difference: record_date minus the satellite midpoint date
            delta = rec_d - mid_d
            row[2] = delta.days
            update_count += 1
            
        except Exception as e:
            row[2] = None
            missing_dates_count += 1
            
        cur.updateRow(row)

# Clear cache to push modifications directly to Pro display
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Midpoint day offset calculation complete.")
print(f"-> Calculated intervals: {update_count:,} rows")
print(f"-> Records skipped due to NULLs: {missing_dates_count:,} rows")
print("="*60)

In [ ]:
#calculate midpoint of MIN_pre_bd_min and MIN_bd_min
import arcpy
import os
from datetime import datetime

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE DATE FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "mid_prebd_MIN_bd_min" not in existing_fields:
    print("Step 1: Adding 'mid_prebd_MIN_bd_min' DATE field...")
    arcpy.management.AddFields(target_fc, [["mid_prebd_MIN_bd_min", "DATE", "mid_prebd_MIN_bd_min"]])
else:
    print("Step 1: Field already exists. Proceeding to calculation...")

# --- 2. CALCULATE PRE-BURN MIDPOINT DATE ---
print("Step 2: Calculating chronological midpoint between pre-burn start and burn start...")
update_count = 0
skipped_count = 0

fields = ["MIN_prebd_min_corrected", "MIN_bd_min_corrected", "mid_prebd_MIN_bd_min"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        pre_start_raw  = row[0]
        burn_start_raw = row[1]
        
        # Check for missing/empty values
        if pre_start_raw in [None, "", 0, "None"] or burn_start_raw in [None, "", 0, "None"]:
            row[2] = None
            skipped_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # 1. Parse YYYYMMDD format safely to datetime objects
            pre_start_str  = str(int(float(pre_start_raw))).strip()
            burn_start_str = str(int(float(burn_start_raw))).strip()
            
            pre_start_date  = datetime.strptime(pre_start_str, "%Y%m%d")
            burn_start_date = datetime.strptime(burn_start_str, "%Y%m%d")
            
            # 2. Compute the exact time delta and divide it by 2
            window_delta = burn_start_date - pre_start_date
            midpoint_date = pre_start_date + (window_delta / 2)
            
            # 3. Truncate hours to midnight to strip any noon timestamps automatically
            row[2] = midpoint_date.replace(hour=0, minute=0, second=0, microsecond=0)
            update_count += 1
            
        except Exception as e:
            row[2] = None
            skipped_count += 1
            
        cur.updateRow(row)

# Flush workspace cache to push updates directly to ArcGIS Pro
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Pre-burn midpoint date compilation complete.")
print(f"-> Populated flat calendar midpoints: {update_count:,} rows")
print(f"-> Records skipped due to NULL bounds: {skipped_count:,} rows")
print("="*60)

In [ ]:
#Calculate days between record date and midpoing just calculated
import arcpy
import os
from datetime import datetime

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matched_only")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE INT FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "days_mid_prebd_MIN_bd_min" not in existing_fields:
    print("Step 1: Adding 'days_mid_prebd_MIN_bd_min' integer field...")
    arcpy.management.AddFields(target_fc, [["days_mid_prebd_MIN_bd_min", "LONG", "days_mid_prebd_MIN_bd_min"]])
else:
    print("Step 1: Field already exists. Proceeding to calculation...")

# --- 2. CALCULATE DAYS INTERVAL ---
print("Step 2: Calculating days between agency record date and pre-burn midpoint...")
update_count = 0
missing_dates_count = 0

fields = ["record_date", "mid_prebd_MIN_bd_min", "days_mid_prebd_MIN_bd_min"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        record_date   = row[0]
        pre_mid_date  = row[1]
        
        # Check if both fields have valid data
        if record_date is None or pre_mid_date is None:
            row[2] = None
            missing_dates_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # Handle potential datetime/date type mismatches safely
            rec_d = record_date.date() if isinstance(record_date, datetime) else record_date
            mid_d = pre_mid_date.date() if isinstance(pre_mid_date, datetime) else pre_mid_date
                
            # Calculate difference: record_date minus the pre-burn midpoint date
            delta = rec_d - mid_d
            row[2] = delta.days
            update_count += 1
            
        except Exception as e:
            row[2] = None
            missing_dates_count += 1
            
        cur.updateRow(row)

# Clear cache to push modifications directly to Pro display
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"SUCCESS. Pre-burn midpoint day offset calculation complete.")
print(f"-> Calculated intervals: {update_count:,} rows")
print(f"-> Records skipped due to NULLs: {missing_dates_count:,} rows")
print("="*60)

In [ ]:
#I will use the midpoint of MIN_pre_bd_min_corrected and MAX_bd_min_corrected to get the month of occurence

import arcpy
import os
from datetime import datetime

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matching_complete")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE DATE FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "midpoint_date" not in existing_fields:
    print("Step 1: Adding 'midpoint_date' DATE field to the master dataset...")
    arcpy.management.AddFields(target_fc, [["midpoint_date", "DATE", "midpoint_date"]])
else:
    print("Step 1: 'midpoint_date' field already exists. Proceeding to calculation...")

# --- 2. CALCULATE FULL WINDOW MIDPOINT DATE ---
print("Step 2: Processing master records using YYYYMMDD formats...")
update_count = 0
skipped_count = 0

# Using the exact fields containing the YYYYMMDD data strings/numbers
fields = ["MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", "midpoint_date"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        pre_start_raw = row[0]
        end_plus8_raw = row[1]
        
        # Guard rail against missing or unpopulated string/numeric bounds
        if pre_start_raw in [None, "", 0, "None", 0.0] or end_plus8_raw in [None, "", 0, "None", 0.0]:
            row[2] = None
            skipped_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # Parse YYYYMMDD formats safely (stripping decimals if they were stored as floats)
            pre_start_str = str(int(float(pre_start_raw))).strip()
            end_plus8_str = str(int(float(end_plus8_raw))).strip()
            
            # Convert the 8-digit strings to datetime objects
            pre_start_date = datetime.strptime(pre_start_str, "%Y%m%d")
            end_plus8_date = datetime.strptime(end_plus8_str, "%Y%m%d")
            
            # Compute the precise halfway mark of the total tracking lifecycle
            full_delta = end_plus8_date - pre_start_date
            midpoint_calc = pre_start_date + (full_delta / 2)
            
            # Drop fractional hours to store a clean, flat calendar date at midnight
            row[2] = midpoint_calc.replace(hour=0, minute=0, second=0, microsecond=0)
            update_count += 1
            
        except Exception as e:
            row[2] = None
            skipped_count += 1
            
        cur.updateRow(row)

# Clear the database cache to make sure the updates instantly refresh in the Pro Map View
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"MASTER DATASET UPDATED SUCCESSFULLY")
print(f"-> Field 'midpoint_date' calculated: {update_count:,} rows")
print(f"-> Records skipped (Missing/Null boundaries): {skipped_count:,} rows")
print("="*60)

In [ ]:
#get midpoint_month
import arcpy
import os

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
target_fc = os.path.join(gdb, "SEFM_events_94_24_matching_complete")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- 1. ENSURE THE MONTH FIELD EXISTS ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if "midpoint_month" not in existing_fields:
    print("Step 1: Adding 'midpoint_month' SHORT INTEGER field to the master dataset...")
    # SHORT type 
    arcpy.management.AddFields(target_fc, [["midpoint_month", "SHORT", "midpoint_month"]])
else:
    print("Step 1: 'midpoint_month' field already exists. Proceeding to calculation...")

# --- 2. EXTRACT MONTH NUMBER FROM MIDPOINT DATE ---
print("Step 2: Extracting numeric months from 'midpoint_date'...")
update_count = 0
skipped_count = 0

fields = ["midpoint_date", "midpoint_month"]

with arcpy.da.UpdateCursor(target_fc, fields) as cur:
    for row in cur:
        mid_date = row[0]
        
        # Guard rail against rows that don't have a calculated midpoint date
        if mid_date is None:
            row[1] = None
            skipped_count += 1
            cur.updateRow(row)
            continue
            
        try:
            # Extract the integer month directly from the datetime object (.month returns 1-12)
            row[1] = int(mid_date.month)
            update_count += 1
            
        except Exception as e:
            row[1] = None
            skipped_count += 1
            
        cur.updateRow(row)

# Clear the database cache to make sure the updates instantly refresh in the Pro Map View
arcpy.management.ClearWorkspaceCache(gdb)

print("\n" + "="*60)
print(f"MASTER MONTH EXTRACTION COMPLETE.")
print(f"-> Field 'midpoint_month' populated: {update_count:,} rows")
print(f"-> Records skipped (Null midpoint dates): {skipped_count:,} rows")
print("="*60)